# Lets . . . tis a keyword

### Logistics
* PS3 is due on Monday
* The project is available and due next Friday at midnight
    * Tomorrow's recitation will be going over the project
* Teaching assignments:
    * First group will be going this Thursday
    * Second group will be going next Thursday
    * If you don't have a group, LET ME KNOW

### The plan for this week
* Monday: 
    * Review of OOP and Pattern Matching
    * Abstract/Concrete Syntax
    * Intro to inference rules
* Tuesday: <-- YOU ARE HERE
    * Inference rules continued
    * Introduction to Lettuce 
* Recitation: Project overview
* Thursday:
    * First student lecture!
    * Discussing variables and the environment

### A Rough Outline
* Inference Rules (coninued)
* Judgment Derivation
* Aside: bring it to the PL world
* Programs in Lettuce
* Scala Vals

# Review: Inference Rules

## Terminology
* Inference rules describe the relation between a set of judgments
    * take the following form: its a line with stuff below it and above it
    
$$\begin{array}{c}
premise_1,\;\;\;\dots,\;\;\;premise_n
\\ \hline
conclusion
\end{array}\texttt{rule-name}$$

* Conclusion: the conclusion is listed below the line and is read first
* Premise: premise are written above the line. An inference rule can have 0-or-many premise
    * Axiom: an inference rule without any premise is called an axiom

## Example: Delete
* Delete an element of the list
    * given a list `l`
    * and a number `n`
    * Delete the first found instance of number `n`
    * Dealers choice:
        * You can decide what to do if the value doesn't exist in the list
        * You can assume the list is ordered, or not. Please state you assumptions clearly
    
* Concrete Grammar **`s`**
    * NOTE: we're including errors now

$$\begin{array}{lll}
v&\Rightarrow& l \\
&|& ERROR \\
\\
l & \Rightarrow & X \\
&|& n\;\rightarrow\;l \\
\\
&&n \in \mathbb{N}
\end{array}$$

* Judgment form: `l delete n = v`

* Choices:
    * Assume the list is not ordered
    * Throw an error if the the data is not found
 

### Where we left off yesterday

$$\newcommand\SemRule[3]{\begin{array}{c}#1\\\hline#2\end{array}\texttt{#3}}$$
$$\SemRule{}{X\;delete\;n=ERROR}{delete-empty}$$<br /><br /><br />
$$\SemRule{n=n_{head}}{n_{head}\rightarrow l_{tail}\;delete\;n=l_{tail}}{delete-head}$$<br /><br /><br />
$$\SemRule{n \neq n_{head},\;\;\;l_{tail}\;delete\;n=l_{tailNew}}{n_{head}\rightarrow l_{tail}\;delete\;n=n_{head}\rightarrow l_{tailNew}}{delete-from-tail}$$<br /><br /><br />

We're reasonably happy with this . . . but what about our Error case?

<details>
  <summary>We need a rule that looks something like this:</summary>
  
  $$\SemRule{n \neq n_{head},\;\;\;l_{tail}\;delete\;n= ERROR}{n_{head}\rightarrow l_{tail}\;delete\;n= ERROR}{delete-from-tail-error}$$<br /><br /><br />
</details>

In [ ]:
// AS CODE
// RETURN ERROR

sealed trait Value
case object MySpecialError extends Value
sealed trait MyList extends Value {
    def delete(n: Double): Value = {
        val l = this
        val v = {
            ???
        }
        v
    }
}
case object EmptyList extends MyList
case class ListNode(n_head: Double, l_tail: MyList) extends MyList

val t0 = EmptyList
val t1 = ListNode(10, t0)
val t2 = ListNode(5, t1)
val t3 = ListNode(5, ListNode(10, ListNode(15, t0)))
val t4 = ListNode(5, ListNode(7, ListNode(10, ListNode(15, t0))))


// UNCOMMENT WHEN READY TO TEST
assert((t0 delete 3155) == MySpecialError)
assert((t4 delete 7) == t3)
assert((t3 delete 15) == t2)
assert((t2 delete 5) == t1)
assert((t1 delete 10) == t0)

// use if returning error
assert((t4 delete 3155) == MySpecialError)

// // use if throwing error
// try {
//     t4 delete 3155
//     println("shouldn't see this message")
// } catch {
//     case _: Throwable => assert(true)
// }

If we're feeling a bit lazy, we don't actually have to percolate the error all the way back up.  We can simply throw one of Scala's exceptions, which overrides everything.

In [ ]:
// AS CODE
// THROW ERROR

sealed trait Value
case object MySpecialError extends Value
sealed trait MyList extends Value {
    def delete(n: Double): Value = {
        val l = this
        val v = {
            l match {
                case EmptyList => throw new Exception("Tried deleting an element from a list which doesn't contain it.")  // throw some exception and be lazy
                case ListNode(n_head, l_tail) if n == n_head => l_tail
                case ListNode(n_head, l_tail) if n != n_head => l_tail delete n match {
                    case l_tailNew: MyList => ListNode(n_head, l_tailNew)
                    // don't need to capture error, it's thrown!
                }
                case _ => ???
            }
        }
        v
    }
}
case object EmptyList extends MyList
case class ListNode(n_head: Double, l_tail: MyList) extends MyList

val t0 = EmptyList
val t1 = ListNode(10, t0)
val t2 = ListNode(5, t1)
val t3 = ListNode(5, ListNode(10, ListNode(15, t0)))
val t4 = ListNode(5, ListNode(7, ListNode(10, ListNode(15, t0))))


// UNCOMMENT WHEN READY TO TEST
assert((t4 delete 7) == t3)
assert((t3 delete 15) == t2)
assert((t2 delete 5) == t1)
assert((t1 delete 10) == t0)

// // use if returning error
// assert((t4 delete 3155) == MySpecialError)

// use if throwing error
try {
    t4 delete 3155
    println("shouldn't see this message")
} catch {
    case _: Throwable => assert(true)
}

In [ ]:
// AS CODE
// RETURN ERROR

sealed trait Value
case object MySpecialError extends Value
sealed trait MyList extends Value {
    def delete(n: Double): Value = {
        this match {
            case EmptyList => MySpecialError
            case ListNode(n_head, l_tail) if n == n_head => l_tail
            case ListNode(n_head, l_tail) if n != n_head => l_tail delete n match {
                case MySpecialError => MySpecialError
                case l_tailNew: MyList => ListNode(n_head, l_tailNew)
            }
            case _ => ???
        }
    }
}
case object EmptyList extends MyList
case class ListNode(n_head: Double, l_tail: MyList) extends MyList

val t0 = EmptyList
val t1 = ListNode(10, t0)
val t2 = ListNode(5, t1)
val t3 = ListNode(5, ListNode(10, ListNode(15, t0)))
val t4 = ListNode(5, ListNode(7, ListNode(10, ListNode(15, t0))))


// UNCOMMENT WHEN READY TO TEST
assert((t4 delete 7) == t3)
assert((t3 delete 15) == t2)
assert((t2 delete 5) == t1)
assert((t1 delete 10) == t0)

// use if returning error
assert((t4 delete 3155) == MySpecialError)

// // use if throwing error
// try {
//     t4 delete 3155
//     println("shouldn't see this message")
// } catch {
//     case _: Throwable => assert(true)
// }

In [ ]:
// AS CODE
// THROW ERROR

sealed trait Value
case object MySpecialError extends Value
sealed trait MyList extends Value {
    def delete(n: Double): Value = {
        this match {
            case EmptyList => ???  // throw some error and be lazy
            case ListNode(n_head, l_tail) if n == n_head => l_tail
            case ListNode(n_head, l_tail) if n != n_head => l_tail delete n match {
                case l_tailNew: MyList => ListNode(n_head, l_tailNew)
                // don't need to caputre error, it's thrown!
            }
            case _ => ???
        }
    }
}
case object EmptyList extends MyList
case class ListNode(n_head: Double, l_tail: MyList) extends MyList

val t0 = EmptyList
val t1 = ListNode(10, t0)
val t2 = ListNode(5, t1)
val t3 = ListNode(5, ListNode(10, ListNode(15, t0)))
val t4 = ListNode(5, ListNode(7, ListNode(10, ListNode(15, t0))))


// UNCOMMENT WHEN READY TO TEST
assert((t4 delete 7) == t3)
assert((t3 delete 15) == t2)
assert((t2 delete 5) == t1)
assert((t1 delete 10) == t0)

// // use if returning error
// assert((t4 delete 3155) == MySpecialError)

// use if throwing error
try {
    t4 delete 3155
    println("shouldn't see this message")
} catch {
    case _: Throwable => assert(true)
}

### Implicit equality
* I will keep it explicit most of the time, like above
* But delete-head can be made an axiom
* NOTE now: $n$ and $n$ are the same. there is no $n_{head}$

$$\newcommand\SemRule[3]{\begin{array}{c}#1\\\hline#2\end{array}\texttt{#3}}$$
$$\SemRule{}{X\;delete\;n=ERROR}{delete-empty}$$<br /><br /><br />
$$\SemRule{}{n\rightarrow l_{tail}\;delete\;n=l_{tail}}{delete-head}$$<br /><br /><br />
$$\SemRule{n \neq n_{head},\;\;\;l_{tail}\;delete\;n= ERROR}{n_{head}\rightarrow l_{tail}\;delete\;n= ERROR}{delete-from-tail-error}$$<br /><br /><br />
$$\SemRule{n \neq n_{head},\;\;\;l_{tail}\;delete\;n=l_{tailNew}}{n_{head}\rightarrow l_{tail}\;delete\;n=n_{head}\rightarrow l_{tailNew}}{delete-from-tail}$$<br /><br /><br />

In [ ]:
sealed trait Value
case object MySpecialError extends Value
sealed trait MyList extends Value {
    def delete(n: Double): Value = {
        this match {
            case EmptyList => MySpecialError
            // WHAT HAPPENED HERE?
            case ListNode(n, l_tail) => l_tail
            case ListNode(n_head, l_tail) if n != n_head => l_tail delete n match {
                case MySpecialError => MySpecialError
                case l_tailNew: MyList => ListNode(n_head, l_tailNew)
            }
            case _ => ???
        }
    }
}
case object EmptyList extends MyList
case class ListNode(n_head: Double, l_tail: MyList) extends MyList

val t0 = EmptyList
val t1 = ListNode(10, t0)
val t2 = ListNode(5, t1)
val t3 = ListNode(5, ListNode(10, ListNode(15, t0)))
val t4 = ListNode(5, ListNode(7, ListNode(10, ListNode(15, t0))))


// UNCOMMENT WHEN READY TO TEST
val given = ListNode(5, ListNode(7, ListNode(10, ListNode(15, EmptyList))))
val value = 10
val found = given delete value
val expected = ListNode(5, ListNode(7, ListNode(15, EmptyList)))

println(s"GIVEN\n\t$given\nDELETE\n\t$value\nExpected\n\t$expected\nFOUND\n\t$found")

// assert((t4 delete 7) == t3)
// assert((t3 delete 15) == t2)
// assert((t2 delete 5) == t1)
// assert((t1 delete 10) == t0)

// // use if returning error
// assert((t4 delete 3155) == MySpecialError)


# Q&A?
???

## Example: Judgment Derivation
* Operational Semantic::
* Grammar:

$$\begin{array}{lll}
v&\Rightarrow& l \\
&|& ERROR \\
\\
l & \Rightarrow & X \\
&|& n\;\rightarrow\;l \\
\\
&&n \in \mathbb{N}
\end{array}$$


* Judgment Form: `l delete n = v`
* Inference rules:

$$\newcommand\SemRule[3]{\begin{array}{c}#1\\\hline#2\end{array}\texttt{#3}}$$
$$\SemRule{}{X\;delete\;n=ERROR}{delete-empty}$$<br /><br /><br />
$$\SemRule{n=n_{head}}{n_{head}\rightarrow l_{tail}\;delete\;n=l_{tail}}{delete-head}$$<br /><br /><br />
$$\SemRule{n \neq n_{head},\;\;\;l_{tail}\;delete\;n=l_{tailNew}}{n_{head}\rightarrow l_{tail}\;delete\;n=n_{head}\rightarrow l_{tailNew}}{delete-from-tail}$$<br /><br /><br />
$$\SemRule{n \neq n_{head},\;\;\;l_{tail}\;delete\;n= ERROR}{n_{head}\rightarrow l_{tail}\;delete\;n= ERROR}{delete-from-tail-error}$$<br /><br /><br />

* We have the operational semanti above
* Let us choice a valid list `l`: `5 -> 7 -> 10 -> X`
    * Sentence Derivation shows the sentence is valid
        * l
        * $\Rightarrow$ n $\rightarrow$ l
        * $\Rightarrow$ 5 $\rightarrow$ l
        * $\Rightarrow$ 5 $\rightarrow$ n $\rightarrow$ l
        * $\Rightarrow$ 5 $\rightarrow$ 7 $\rightarrow$ l
        * $\Rightarrow$ 5 $\rightarrow$ 7 $\rightarrow$ n $\rightarrow$ l
        * $\Rightarrow$ 5 $\rightarrow$ 7 $\rightarrow$ 10 $\rightarrow$ l
        * $\Rightarrow$ 5 $\rightarrow$ 7 $\rightarrow$ 10 $\rightarrow$ X
* Let us choose a valid number: `7`
    * it is a natural number
    * deal with it

* we want to find `5 -> 7 -> 10 -> X delete 7 = ???`
* what do we expect? `5 -> 10 -> X`

$$\SemRule{}{X\;delete\;n=ERROR}{delete-empty}$$<br /><br /><br />
$$\SemRule{n=n_{head}}{n_{head}\rightarrow l_{tail}\;delete\;n=l_{tail}}{delete-head}$$<br /><br /><br />
$$\SemRule{n \neq n_{head},\;\;\;l_{tail}\;delete\;n=l_{tailNew}}{n_{head}\rightarrow l_{tail}\;delete\;n=n_{head}\rightarrow l_{tailNew}}{delete-from-tail}$$<br /><br /><br />
$$\SemRule{n \neq n_{head},\;\;\;l_{tail}\;delete\;n= ERROR}{n_{head}\rightarrow l_{tail}\;delete\;n= ERROR}{delete-from-tail-error}$$<br /><br /><br />

LIVE
~~~
            7 = 7
            ----------------------------------(delete-head)
7 != 5,     7 -> 10 -> X delete 7 = 10 -> X
-------------------------------------------------------------------(delete-from-tail)
                  5 -> 7 -> 10 -> X delete 7 = 5 -> 10 -> X
~~~

## Try it yourself: Judgment Derivation

* Operational Semantic::
* Grammar:

$$\begin{array}{lll}
v&\Rightarrow& l \\
&|& ERROR \\
\\
l & \Rightarrow & X \\
&|& n\;\rightarrow\;l \\
\\
&&n \in \mathbb{N}
\end{array}$$


* Judgment Form: `l delete n = v`
* Inference rules:

$$\newcommand\SemRule[3]{\begin{array}{c}#1\\\hline#2\end{array}\texttt{#3}}$$
$$\SemRule{}{X\;delete\;n=ERROR}{delete-empty}$$<br /><br /><br />
$$\SemRule{n=n_{head}}{n_{head}\rightarrow l_{tail}\;delete\;n=l_{tail}}{delete-head}$$<br /><br /><br />
$$\SemRule{n \neq n_{head},\;\;\;l_{tail}\;delete\;n= ERROR}{n_{head}\rightarrow l_{tail}\;delete\;n= ERROR}{delete-from-tail-ERROR}$$<br /><br /><br />
$$\SemRule{n \neq n_{head},\;\;\;l_{tail}\;delete\;n=l_{tailNew}}{n_{head}\rightarrow l_{tail}\;delete\;n=n_{head}\rightarrow l_{tailNew}}{delete-from-tail}$$<br /><br /><br />

* We have the operational semantics above

* Let us choice a valid list `l`: `5 -> 7 -> X`
    * Sentence Derivation shows the sentence is valid
        * l
        * $\Rightarrow$ n $\rightarrow$ l
        * $\Rightarrow$ 5 $\rightarrow$ l
        * $\Rightarrow$ 5 $\rightarrow$ n $\rightarrow$ l
        * $\Rightarrow$ 5 $\rightarrow$ 7 $\rightarrow$ l
        * $\Rightarrow$ 5 $\rightarrow$ 7 $\rightarrow$ X
* Let us choose a valid number that doesn't exist in the list `l`: `10`
    * it is a natural number
    * deal with it
* judgment: `5 -> 7 -> X delete 10 = ERROR`
* Do you agree the judgment should hold true?
    * If so, do a judgment derivation to demonstrate it as true
    * Else, explain why the judgment is incorrect

~~~

_P0_,     ...
-------------------------------------------------------------------(rule-name)
                  _YOUR_JUDGMENT_HERE_
~~~

# Aside: bring it to the PL world
* in yesterdays lecture I have a LOT of information on program evaluation for a custom language of lists
* Check it out if you're interested
* it's too deep for us this semester

## The punchline for this semester
* if we have expressions named `e`
* and values named `v`
* then we have 3 choices of judgement forms

### Big-step/natural interpreters
#### `eval(e) = v`
* expression e evaluates to a value v
* we'll likely use this all semester
* e.g.
    * `eval(1 + 2 * 3) = 7`
    
#### `e` $\Downarrow$ `v`
* expression e evaluates to a value v
* another way to write `eval(e) = v`
* e.g. 
    * `1 + 2 * 3` $\Downarrow$ `7`

### small-step interpreters
#### `e` $\longrightarrow$* `v`
* expression e steps to a value v in 0-or-many-steps
* a great topic, I likely won't get to cover this summer
* supported by 
    * `e` $\longrightarrow$ `e'`
    * expression `e` steps to a new expression `e'` in exactly one step
* e.g.
    * `1 + 2 * 3` $\longrightarrow$* `7`
        * because
        * `1 + 2 * 3` 
        * $\longrightarrow$ `1 + 6`
        * $\longrightarrow$ `7`

# Programs
* We will now pivot to the lettuce programming language
* Our goal is to author our own interpreter
* It shall implement a language similar to <a href="https://ocaml.org/">OCaml</a>
* We define the language of our programs
    * Using inductive logic in grammars
        * for concrete syntax
* We define the semantics of features in the language
    * Using inductive logic as inference rules
* We code it in Scala
    * We re-write the grammar as an abstract syntax
    * We code the class hierarchy
    * We code the evaluation methods 
    * And any helpers needed

# Scala Vals
* Let's pivot
* consider the language of lets
* first a somewhat silly exercise 
    * and yet it's actually quite complicated in code (at least the first time you see it)

In [ ]:
// what's the expected value
1

In [ ]:
// what's the expected value
1 + 2

In [ ]:
// what's the expected value
1 + (2 / 2)

In [ ]:
// what's the expected value
{ 
    def f(n: Int): Int = n * 2
    f(22)  // 44
} + {
    def g(n: Int): Int = n / 5
    g(100)  // 20
}

In [ ]:
// // what's the expected value
// x
// // ERRROR

In [ ]:
// what's the expected value
val x = 1
x

In [ ]:
// what's the expected value
val x = 1 + 2 / 2
x

In [ ]:
// what's the expected value
val x = 1 + 1
x

In [ ]:
// what's the expected value
val x = 2
x

In [ ]:
// what's the expected value
2

In [ ]:
// what's the expected value
val x = {
    val y = 2
    y + 1
}
x

In [ ]:
// // what's the expected value
val x = {
    val y = 2
    y + 1
}
x + y
// // ERROR

In [ ]:
// what's the expected value
val x = {
    val x = 2
    x + 1
}
x

In [ ]:
// // what's the expected value
val x = {
    val x = x + 1
    x + 1
}
x
// // ERROR

In [ ]:
// what's the expected value
val x = {
    val x = {
        val x = 1
        x + 1
    }
    x + 1
}
x

# Lettuce Lets
* We introduce a language called lettuce
* For now it can look like:

$$\begin{array}{lrl}
e & \Rightarrow & v \\
&|& e_1 + e_2 & \texttt{we can add multiple expressions} \\
&|& e_1\;/\;e_2 & \texttt{we can divide multiple expressions} \\
&|& (e) \\
\\
v & \Rightarrow & n \\
\\
&&& \texttt{n is a number}\\
\end{array}$$

* What would you expect is the value of the following lettuce program

~~~
1
~~~

* What would you expect is the value of the following lettuce program

~~~
1 + 2
~~~

* What would you expect is the value of the following lettuce program

~~~
1 + 2/2
~~~

* What would you expect is the value of the following lettuce program

~~~
x
~~~